In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
import kagglehub


### Step 1: Download ORL Dataset

In [ ]:
# Download latest version
path = kagglehub.dataset_download("kasikrit/att-database-of-faces")
print("Path to dataset files:", path)


Path to dataset files: /home/habiba/.cache/kagglehub/datasets/kasikrit/att-database-of-faces/versions/2


### Step 2: Generate the Data Matrix and the Label vector

In the AT&T Face Dataset, a subject refers to a person. The dataset includes 40 different people, and each one is stored in a separate folder
Reading all 400 images (10 per person × 40 people)
Storing them in images (shape: 400, 112, 92)
Assigning a label (person ID) from 1 to 40 to each image

In [19]:
images = []
Y = []
# Loop through subjects (s1 to s40)
for subject_id in range(1, 41): 
    subject_folder = os.path.join(path, f's{subject_id}')
    
    # Loop through each of the 10 images for the subject
    for img_number in range(1, 11):  # 1.pgm to 10.pgm
        img_path = os.path.join(subject_folder, f'{img_number}.pgm')
        
        # Read image 
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        # Append image and label
        images.append(img)
        Y.append(subject_id)  # Use subject_id as the label

images = np.array(images)        
Y = np.array(Y) 
D = images.reshape(400, -1)
D.shape

print("Loaded dataset shape:", D.shape)
print("Y shape:", Y.shape)


Loaded dataset shape: (400, 10304)
Y shape: (400,)


### Step 3: Split the Dataset into Training and Test sets

In [20]:
# Select training and testing rows
D_train = D[::2]  # odd-numbered rows
D_test  = D[1::2] # even-numbered rows 

y_train = Y[::2]
y_test  = Y[1::2]


### PCA Implementation

In [26]:
class PCA:
    def __init__(self, D_train, D_test, y_train, y_test):
        self.D_train = D_train
        self.D_test = D_test
        self.y_train = y_train
        self.y_test = y_test
        self.D_centered =[]
        self.eigenvalues=[]
        self.eigenvectors=[]
    
    def compCov(self):
        """Compute the covariance matrix"""
        mean_face = np.mean(self.D_train, axis=0)
        self.D_centered = self.D_train - mean_face
        Cov = self.D_centered @ self.D_centered.T
        return Cov
    
    def compEig(self,Cov):
        """Compute the eigenvalues and the eigenvectors"""
        eigenvalues , eigenvectors =np.linalg.eigh(Cov)
        eigenvectors = eigenvectors / np.linalg.norm(eigenvectors, axis=0)
        self.eigenvalues = eigenvalues[::-1]
        self.eigenvectors = eigenvectors[:, ::-1]
    
    def selectComp(self,alpha):
        """Select the first i eigenvectors based on the explained variance"""
        sum=0
        i=0
        eigsum =np.sum(self.eigenvalues)
        for val in self.eigenvalues:
            sum+=val
            i+=1
            if(val/eigsum) == alpha:
                return self.eigenvectors[: , :i]
        
    def project(self,alpha):
        """Project the data on the eigenvectors"""
        Cov =self.compCov()
        self.compEig(Cov)
        eigvec = self.selectComp(alpha)
        proj = self.D_train.T @ eigvec

    
x =PCA(D_train, D_test, y_train,y_test)
x.project(0.95)
 

ValueError: matmul: Input operand 1 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)

## Unsupervised Clustering
### K-Means Clustering

### K-Means Clustering Evaluation

### Gaussian Mixture Model Clustering

### Gaussian Mixture Model Clustering Evaluation

### Bonus